# [센서 실습 8] PIR 인체 감지 센서 – HC-SR501

인체 감지 센서 **HC-SR501**의 출력을 읽어 사람의 움직임을 감지하고, 마지막 움직임 후 일정 시간 켜져 있는 **자동 조명 타이머**를 만듭니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
PIR(Passive InfraRed)은 스스로 빛을 내지 않고, 사람 몸에서 나오는 **원적외선(약 10µm)의 변화**를 감지합니다.
- 센서 안에 감지 소자가 2개 있어, 사람이 지나가며 두 소자에 들어오는 적외선 양이 **달라질 때** 신호 발생
- 가만히 있는 사람은 잘 감지하지 못함 (변화가 없으므로)
- 앞의 흰 반구(**프레넬 렌즈**)가 넓은 범위의 적외선을 모아 줌
- 감지되면 출력이 HIGH가 되고, 모듈의 가변저항으로 정한 **유지 시간**(약 3초~5분) 동안 HIGH 유지
- 전원을 켠 뒤 **약 1분 동안은 안정화** 시간이 필요함

### 부품 사양
| 항목 | 내용 |
|---|---|
| 감지 거리 | 3 ~ 7 m (가변저항 조절) |
| 감지 각도 | 약 110° |
| 유지 시간 | 약 3초 ~ 5분 (가변저항) |
| 출력 | 디지털 3.3V (감지 시 HIGH) |
| 전원 | 4.5 ~ 20V (5V 사용) |

## 2. 라즈베리파이 연결 방법
**준비물**: HC-SR501, 점퍼선 3개 (+ LED 또는 릴레이)

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| HC-SR501 VCC | 5V (2번 핀) | 전원은 5V |
| HC-SR501 GND | GND (6번 핀) |  |
| HC-SR501 OUT | GPIO17 (11번 핀) | 출력은 3.3V라 바로 연결 가능 |
| LED(+) – 330Ω | GPIO18 (12번 핀) | 조명 역할 (선택) |

모듈의 점퍼를 **H(반복 트리거)** 로 두면, 사람이 계속 움직이는 동안 유지 시간이 계속 연장됩니다.

**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
PIR 출력은 이미 0/1이므로 변환식 대신, **마지막 움직임 후 N초 동안 조명을 켜 두는 타이머** 함수를 공통으로 씁니다.

In [ ]:
def light_timer(motion_series, hold_s, dt):
    """움직임(0/1) 목록 → 조명 상태(0/1). 마지막 움직임 후 hold_s 초 동안 켜 둠"""
    remain, out = 0.0, []
    for m in motion_series:
        remain = hold_s if m else max(0.0, remain - dt)
        out.append(int(remain > 0))
    return out

print(light_timer([0, 1, 0, 0, 0, 0], hold_s=2, dt=1))

### 4-2. 실제 센서 클래스
`gpiozero.MotionSensor`가 PIR 센서용 클래스를 제공합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealPIR:
    def __init__(self, pin=17):
        from gpiozero import MotionSensor
        self.dev = MotionSensor(pin)
        print("PIR 안정화 중... (약 60초)")

    def read(self):
        return {"motion": int(self.dev.motion_detected)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 복도에 설치, 5분 관찰 | 자동 조명 |
| 사람 통과 | 30초, 95초, 180~200초(서성임), 260초 | 가끔 지나가는 복도 |
| 센서 유지 시간 | 마지막 움직임 후 3초 HIGH | 가변저항 최소 설정 |
| 오감지 | 처음 10초 동안 무작위 HIGH | 전원 투입 직후 안정화 구간 |

In [ ]:
class VirtualPIR:
    HOLD = 3.0                                       # 센서 자체 유지 시간(초)

    def __init__(self, dt=1.0, seed=8):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)
        self.moves = [(30, 34), (95, 99), (180, 200), (260, 265)]   # 사람이 움직이는 구간(초)
        self.last_move = -999.0

    def read(self):
        moving = any(a <= self.t < b for a, b in self.moves)
        if moving:
            self.last_move = self.t
        out = int(self.t - self.last_move < self.HOLD)
        if self.t < 10 and self.rng.random() < 0.4:   # 안정화 전 오감지
            out = 1
        self.t += self.dt
        return {"motion": out}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualPIR(dt=1.0)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
1초 간격으로 5분 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 300               # 측정 횟수
INTERVAL = 1.0               # 측정 간격(초)

sensor = RealPIR() if USE_REAL_SENSOR else VirtualPIR(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. 전원 투입 후 **처음 10초는 안정화 구간**으로 보고 무시합니다.
2. 마지막 움직임 후 **20초 동안 조명**을 켜 둡니다.
3. 항상 켜 두었을 때와 비교해 **절약률**을 계산합니다.

In [ ]:
WARMUP, HOLD = 10, 20
df.loc[df["time_s"] < WARMUP, "motion"] = 0
df["light_on"] = light_timer(df["motion"], hold_s=HOLD, dt=INTERVAL)
entries = (df["motion"].diff() == 1).sum()
on_ratio = df["light_on"].mean()
print(f"사람 감지 횟수: {entries}회")
print(f"조명 켜진 시간: {df['light_on'].sum() * INTERVAL:.0f}초 / 전체 {N_SAMPLES * INTERVAL:.0f}초")
print(f"항상 켜 두는 것보다 {100 * (1 - on_ratio):.0f}% 절약")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
ax[0].step(df["time_s"], df["motion"], "k-", where="post"); ax[0].set_ylabel("PIR 출력")
ax[0].axvspan(0, WARMUP, color="lightgray", alpha=0.6); ax[0].text(1, 0.5, "안정화", fontsize=9)
ax[0].set_title("HC-SR501 인체 감지와 자동 조명")
ax[1].fill_between(df["time_s"], df["light_on"], step="post", color="lightgray")
ax[1].step(df["time_s"], df["light_on"], "k-", where="post"); ax[1].set_ylabel("조명")
ax[1].set_xlabel("시간 (초)")
for a in ax: a.set_yticks([0, 1]); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "PIR인체감지센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "HC-SR501",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/motion", payload)
# client.disconnect()

## 직접 해 보기
- 조명 유지 시간 `HOLD`를 5초, 60초로 바꾸면 절약률이 어떻게 달라지나요?
- 밤(조도 낮음)일 때만 조명을 켜도록 4번 조도 센서와 조건을 합쳐 보세요.
- 외출 모드에서 움직임이 감지되면 '침입 감지' 알림 메시지를 만들어 보세요.